# Cost functions / One cell at a time

Put the concepts learned through slide 39 into practice. Work from top to bottom: **read the explanation, run a short cell, interpret its output**. Start by clicking each cell's run button rather than using Run all.

Compare actual and predicted values for eight cars from a public dataset. Blue dots show actual values, orange crosses show predictions, and purple lines show their errors.

At the end, use Colab sliders and a play button to view the prediction function on the left and the cost function on the right together. Everything runs inside the notebook on a CPU runtime.

Today's question: **How can one number tell us how well this line fits?**

## 0. Setup
The next cell prepares the plotting tools and font. Run it once. Its setup code is collapsed; below it, the main calculations are separated into short steps.

In [ ]:
# @title Prepare plotting tools and font
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import ipywidgets as widgets
from IPython.display import display
from google.colab import output

output.enable_custom_widget_manager()
font_path = Path('/content/NanumGothic-Regular.ttf')
if not font_path.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
BLUE, ORANGE, PURPLE, GREEN = '#1768ad', '#c86600', '#8c439c', '#207454'
print('Setup complete. Continue to the next cell.')

## 1. Load actual observations
Use a car's **weight to predict its fuel economy**. Higher miles per gallon (mpg) means traveling farther on the same amount of fuel.

Source: R. Quinlan (1993), [UCI Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg), [DOI 10.24432/C5859H](https://doi.org/10.24432/C5859H), CC BY 4.0. The download uses the [public seaborn copy](https://github.com/mwaskom/seaborn-data/blob/master/mpg.csv). Sort all 398 observations by weight and select eight evenly spaced actual observations. This selection is for calculation practice, not performance evaluation.

In [ ]:
cars = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv')
ordered = cars.sort_values('weight', kind='stable').reset_index(drop=True)
sample = ordered.iloc[np.rint(np.linspace(0, len(ordered) - 1, 8)).astype(int)][['name', 'weight', 'mpg']].copy()
sample.index = np.arange(1, 9)
display(sample.rename(columns={'name': 'Car', 'weight': 'Weight (lb)', 'mpg': 'Actual fuel economy (mpg)'}))

### Input $x$, actual value $y$, and example count $m$
$x$ is weight in thousands of pounds. For example, 1,613 pounds becomes $x=1.613$. $y$ is actual fuel economy, and $m$ is the number of cars. Run the next three cells in order.

In [ ]:
x = sample['weight'].to_numpy() / 1000
x

In [ ]:
y = sample['mpg'].to_numpy()
y

In [ ]:
m = len(y)
m

### Look at the actual values first
Each blue point is one car. The horizontal axis shows weight; the vertical axis shows actual fuel economy. Describe how fuel economy tends to change as cars get heavier.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y, color=BLUE, label='Actual', s=60)
plt.xlabel('Weight x (1,000 lb)')
plt.ylabel('Fuel economy y (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 2. Predict with one straight line
$$\hat y=f_{w,b}(x)=wx+b$$
The slope $w$ controls the line's steepness, and the intercept $b$ controls its vertical position. Start with $w=-5$ and $b=35$. This is just a candidate line, not a claim that it is the best fit.

In [ ]:
w, b = -5.0, 35.0

In [ ]:
y_pred = w * x + b
y_pred

A prediction does not change the actual target. For each car, place its measured value beside the value our line predicts.

In [ ]:
comparison = pd.DataFrame({'Weight x': x, 'Actual y': y, 'Prediction': y_pred}, index=sample.index)
display(comparison.round(3))

### Compare actual values and predictions by color
Pair each blue dot with the orange cross at the same horizontal position. The closer they are, the more accurate that car's prediction is.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(x, y, color=BLUE, label='Actual', s=65)
plt.scatter(x, y_pred, color=ORANGE, marker='x', label='Prediction', s=75)
plt.plot(x, y_pred, color=ORANGE, alpha=.7)
plt.xlabel('Weight x (1,000 lb)')
plt.ylabel('Fuel economy (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 3. Calculate each car's error
$$e^{(i)}=\hat y^{(i)}-y^{(i)}$$
An overprediction gives a positive error; an underprediction gives a negative error. The purple vertical line's **length** is the error magnitude. Its sign tells us whether the prediction is above or below the actual value.

In [ ]:
errors = y_pred - y
errors

In [ ]:
plt.figure(figsize=(8, 4))
plt.vlines(x, y, y_pred, color=PURPLE, linestyle='--', label='Error magnitude')
plt.scatter(x, y, color=BLUE, label='Actual', s=65)
plt.scatter(x, y_pred, color=ORANGE, marker='x', label='Prediction', s=75)
plt.plot(x, y_pred, color=ORANGE, alpha=.6)
plt.xlabel('Weight x (1,000 lb)')
plt.ylabel('Fuel economy (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

### Why not just add the errors?
Predict 5 too high for one car and 5 too low for another, and their errors add to zero. Both predictions are still wrong. It is like answering two exam questions incorrectly and having the mistakes cancel to give a perfect score.

In [ ]:
np.array([5, -5]).sum()

## 4. Square the errors so their signs cannot cancel
$$\left(\hat y^{(i)}-y^{(i)}\right)^2$$
$(-5)^2=25$ and $5^2=25$. Squaring also gives larger errors more weight: doubling an error makes its square four times larger.

In [ ]:
squared_errors = errors ** 2
squared_errors

In [ ]:
display(pd.DataFrame({'Actual y': y, 'Prediction': y_pred, 'Error': errors, 'Squared error': squared_errors}, index=sample.index).round(3))

## 5. $\sum$ means add the terms one by one
$$\sum_{i=1}^{m}\left(\hat y^{(i)}-y^{(i)}\right)^2$$
$i$ is the car index. Here, add every squared error from car 1 to car 8. Collecting each car's squared error is like adding individual item prices to get a shopping basket's total.

In [ ]:
running_total = np.cumsum(squared_errors)
display(pd.DataFrame({'This squared error': squared_errors, 'Running total': running_total}, index=sample.index).round(3))

### Watch the total grow one car at a time
Move **Cars included** from 0 to 8, or use the play button to advance one at a time. Faint bars show values that have not yet been added.

In [ ]:
# @title Add squared errors one at a time
def show_sum(count):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar(np.arange(1, m + 1), squared_errors, color=GREEN)
    for i, bar in enumerate(bars):
        bar.set_alpha(1 if i < count else .15)
    ax.set(xlabel='Car index i', ylabel='Squared error', title=f'{count} cars included: total squared error = {squared_errors[:count].sum():.3f}')
    ax.set_xticks(np.arange(1, m + 1))
    plt.show()

count_slider = widgets.IntSlider(value=0, min=0, max=m, description='Cars included', style={'description_width': 'initial'})
sum_play = widgets.Play(value=0, min=0, max=m, interval=1100)
sum_link = widgets.jslink((sum_play, 'value'), (count_slider, 'value'))
sum_output = widgets.interactive_output(show_sum, {'count': count_slider})
display(widgets.HBox([sum_play, count_slider]), sum_output)

In [ ]:
sum_squared_error = squared_errors.sum()
sum_squared_error

## 6. Divide by $m$ to take the mean
$$\frac{1}{m}\sum_{i=1}^{m}\left(\hat y^{(i)}-y^{(i)}\right)^2$$
Use the average per car so the score does not grow merely because there are more cars. Duplicating the same eight cars doubles both the sum and the count, leaving their mean unchanged.

In [ ]:
mean_squared_error = sum_squared_error / m
mean_squared_error

## 7. Divide by 2 to obtain cost $J$
$$J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2$$
$\frac{1}{2m}$ means **divide by $m$, then divide by 2**. Here $m=8$, so the denominator is 16. The factor $1/2$ is a convention that simplifies derivatives later. Halving every cost equally does not change which line is best.

In [ ]:
J = mean_squared_error / 2
J

In [ ]:
display(pd.DataFrame({'Calculation step': ['Sum of squared errors', 'Mean: divide by 8', 'Cost J: divide by 2 again'], 'Value': [sum_squared_error, mean_squared_error, J]}))

## 8. Prediction function versus cost function
$f_{w,b}(x)$ is **the predicted fuel economy for one car**. $J(w,b)$ is **one number evaluating the chosen line's predictions across all cars**.

The next cell wraps our preceding calculations in one function. The central calculation takes one line.

In [ ]:
def cost(w, b):
    return np.mean((w * x + b - y) ** 2) / 2

cost(-5, 35)

## 9. $\min_w J(w)$ / Choose only the slope
First fix $b=0$. Then $f_w(x)=wx$, and every candidate line passes through the origin.

**The left horizontal axis is car weight $x$; the right horizontal axis is slope $w$.** One line on the left corresponds to one point on the right cost curve. Calculating that single point's height still uses all eight cars.

The next cell prepares the plots. Its drawing code is collapsed; use the slider cell below it during the lesson.

In [ ]:
# @title Prepare the side-by-side plotting function
def draw_predictions(ax, slope, intercept):
    predicted = slope * x + intercept
    grid = np.linspace(0, 5.5, 100)
    ax.plot(grid, slope * grid + intercept, color=ORANGE)
    ax.vlines(x, y, predicted, color=PURPLE, linestyles='--', alpha=.65)
    ax.scatter(x, y, color=BLUE, label='Actual', s=45)
    ax.scatter(x, predicted, color=ORANGE, marker='x', label='Prediction', s=50)
    ax.set(xlabel='Input x / weight', ylabel='Fuel economy', title=f'Prediction function: {slope:.2f}x + {intercept:.2f}')
    ax.legend()
    ax.grid(alpha=.2)

def compare_fixed(slope, intercept):
    candidates = np.linspace(-20, 20, 201)
    costs = np.array([cost(value, intercept) for value in candidates])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    draw_predictions(axes[0], slope, intercept)
    axes[1].plot(candidates, costs, color=GREEN)
    axes[1].scatter([slope], [cost(slope, intercept)], color=ORANGE, s=80, zorder=3)
    axes[1].set(xlabel='Slope w', ylabel='Cost J across 8 cars', title=f'Fixed b={intercept:.0f}, current J={cost(slope, intercept):.2f}')
    axes[1].grid(alpha=.2)
    plt.show()

### Experiment A / Change only the slope
Change the slope and watch the left line and the right orange point together. Playback tries a sequence of slopes; it is not gradient descent. Selecting an intercept of 45 displays a different cost curve. **Within each curve, the intercept stays fixed.**

In [ ]:
slope_slider = widgets.FloatSlider(value=6, min=-20, max=20, step=.5, description='Slope w', continuous_update=True)
fixed_b = widgets.Dropdown(options=[('Fixed at 0', 0), ('Fixed at 45', 45)], description='Intercept b')
slope_play = widgets.Play(value=6, min=-20, max=20, step=1, interval=700)
slope_link = widgets.jslink((slope_play, 'value'), (slope_slider, 'value'))
fixed_output = widgets.interactive_output(compare_fixed, {'slope': slope_slider, 'intercept': fixed_b})
display(widgets.VBox([widgets.HBox([slope_play, slope_slider]), fixed_b]), fixed_output)

## 10. $\min_{w,b}J(w,b)$ / Choose slope and intercept
Now choose both the slope and vertical position of the line. On the right, the horizontal axis is $w$ and the vertical axis is $b$. **One point on the map represents one line**, and brighter colors indicate lower costs.

Choosing only $b=0$ restricts the search to the map's bottom row. Choosing both parameters allows the entire map. The cost formula is the same, but the available set of lines is different.

In [ ]:
# @title Prepare the two-parameter cost map
w_grid = np.linspace(-20, 20, 121)
b_grid = np.linspace(0, 90, 121)
W, B = np.meshgrid(w_grid, b_grid)
surface = np.mean((W[..., None] * x + B[..., None] - y) ** 2, axis=-1) / 2

def compare_joint(slope, intercept):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    draw_predictions(axes[0], slope, intercept)
    levels = np.geomspace(max(surface.min() * .9, .01), surface.max(), 25)
    contour = axes[1].contourf(W, B, surface, levels=levels, cmap='Blues')
    axes[1].axhline(0, color='black', linestyle='--', linewidth=2)
    axes[1].scatter([slope], [intercept], color=ORANGE, edgecolor='black', s=85, zorder=3)
    axes[1].set(xlabel='Slope w', ylabel='Intercept b', title=f'Current cost J={cost(slope, intercept):.2f} · brighter means lower')
    fig.colorbar(contour, ax=axes[1], label='Cost J')
    plt.show()

### Experiment B / Move two controls separately
First change only the slope, then only the intercept. Finally adjust both to reduce the cost. The goal is to describe **how each control changes the line**, rather than simply guess the best values.

In [ ]:
joint_w = widgets.FloatSlider(value=6, min=-20, max=20, step=.2, description='Slope w', continuous_update=True)
joint_b = widgets.FloatSlider(value=0, min=0, max=90, step=.5, description='Intercept b', continuous_update=True)
joint_output = widgets.interactive_output(compare_joint, {'slope': joint_w, 'intercept': joint_b})
display(widgets.HBox([joint_w, joint_b]), joint_output)

## 11. Compare your line with the minimum cost
The next cell calculates minimum costs for comparison. You do not need to memorize the method yet. For a fair comparison, it also uses the best allowed value when only the slope is free.

In [ ]:
best_w_fixed = (x @ y) / (x @ x)
best_w, best_b = np.linalg.lstsq(np.column_stack([x, np.ones(m)]), y, rcond=None)[0]
display(pd.DataFrame({'Free parameters': ['Choose w; fix b=0', 'Choose both w and b'], 'w': [best_w_fixed, best_w], 'b': [0, best_b], 'Minimum cost': [cost(best_w_fixed, 0), cost(best_w, best_b)]}).round(3))

When both $w,b$ are free, lines with $b=0$ are still available. The achievable minimum cost therefore cannot exceed the minimum with $b=0$ fixed. Depending on the data, the two minimum costs could be equal.

The notation $J(w)$ does not always mean $b=0$. This lab explicitly labels **which value of $b$ is being held fixed**.

### Check your understanding
1. Why is an error negative when a prediction is below the actual value?
2. Why divide the sum of squared errors by 8, and why divide it by 2 afterward?
3. Why does one line on the left correspond to one point on the right cost curve?
4. How does choosing only a slope differ from choosing both slope and intercept?

A lower cost means a better fit to these eight cars. Whether the model also works for new cars will be evaluated separately in a later lesson.